# End-to-End GRPO with KeysAndValues

This notebook demonstrates the full GRPO (Group Relative Policy Optimization) pipeline running entirely on KeysAndValues components, with every memory-heavy step routed through the KV cache.

The pipeline:
1. **Generate** completions (chunked KV-cache decode)
2. **Reward** the completions
3. **Group-relative advantages** (the core of GRPO)
4. **Old log-probs** under the sampling policy (no grad)
5. **Policy gradient** via a GRPO loss `HeadModel` + `LongContextGradientModel` (memory-bounded backward)
6. **Optimizer step**

Steps 1, 4, and 5 all run through the KV cache, so GPU memory stays bounded even for long completions. This runs on CPU with a tiny model.

In [1]:
import torch

from keys_values.config import Config
from keys_values.model import GPT
from keys_values.kvcache.factory import KVCacheFactory
from keys_values.finetune.grpo_loop import grpo_step, compute_group_advantages

/Users/mjerge/Documents/Projects/keys_values/keyval_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/mjerge/Documents/Projects/keys_values/keyval_venv/lib/python3.12/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


## 1. Understanding group-relative advantages

GRPO samples `G` completions per prompt, then normalizes rewards within each group to zero mean and unit std. This is what gives GRPO its name and removes the need for a separate value network.

In [2]:
# Two prompts, 3 completions each. Rewards laid out group-by-group.
rewards = torch.tensor([1.0, 2.0, 6.0,    10.0, 10.0, 10.0])
advantages = compute_group_advantages(rewards, group_size=3)

print("Rewards:    ", rewards.tolist())
print("Advantages: ", [round(a, 3) for a in advantages.tolist()])
print("\nGroup 1: spread rewards -> non-zero advantages (learning signal)")
print("Group 2: identical rewards -> ~zero advantages (no signal, as expected)")

Rewards:     [1.0, 2.0, 6.0, 10.0, 10.0, 10.0]
Advantages:  [-0.756, -0.378, 1.134, 0.0, 0.0, 0.0]

Group 1: spread rewards -> non-zero advantages (learning signal)
Group 2: identical rewards -> ~zero advantages (no signal, as expected)


## 2. Build a small policy model with KV caches

The model needs non-dense KV caches assigned (the `LongContextGradientModel` used in the backward pass requires them). Here we use the `lastrec` (sliding window) policy; `h2o-*` policies work too.

In [3]:
config = Config(
    block_size=512,
    vocab_size=64,
    padded_vocab_size=64,
    n_layer=2,
    n_head=4,
    n_embd=32,
    n_query_groups=2,
    intermediate_size=64,
    rotary_percentage=1,
)

torch.manual_seed(0)
torch.set_default_dtype(torch.float32)

num_prompts = 2
group_size = 4
batch_size = num_prompts * group_size
cache_length = 64

with torch.device("cpu"):
    model = GPT(config)
    model.apply(model._init_weights)

model.assign_kv_caches(
    KVCacheFactory.create(
        gpt_model=model,
        name="lastrec-default",
        max_batch_size=batch_size,
        cache_length=cache_length,
        dtype=torch.float32,
    )
)
print(f"Model: {sum(p.numel() for p in model.parameters()):,} params, {config.n_layer} layers, lastrec cache (length {cache_length})")

Model: 19,136 params, 2 layers, lastrec cache (length 64)


## 3. Define a reward function

In a real setup this would score the decoded text (correctness, format, etc.). Here we use a toy deterministic reward: prefer completions with a higher mean token id. The reward function receives `(prompt_ids, completion_ids)` and returns one scalar per completion.

In [4]:
def reward_fn(prompt_ids, completion_ids):
    # one reward per completion in the batch
    return completion_ids.float().mean(dim=1)

## 4. Run GRPO training steps

Each `grpo_step` call runs the entire pipeline: generate -> reward -> advantages -> old log-probs -> policy gradient -> optimizer step. We sample (temperature 1.0) so completions within a group differ, producing a learning signal.

Note: with one inner update per generation (the TRL default, `num_iterations=1`), the importance ratio is exactly 1.0 at the step, so the *loss value* is near zero even though the *gradient* is non-zero and the policy updates. We therefore also track `advantage_std`, the magnitude of the learning signal within groups.

In [5]:
prompt_ids = torch.randint(0, config.vocab_size, (num_prompts, 8))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(5):
    metrics = grpo_step(
        gpt_model=model,
        prompt_ids=prompt_ids,
        reward_fn=reward_fn,
        optimizer=optimizer,
        group_size=group_size,
        max_new_tokens=8,
        chunk_size=16,
        temperature=1.0,
    )
    print(
        f"step {step}: loss={metrics['loss']:+.4f}  "
        f"mean_reward={metrics['mean_reward']:.3f}  "
        f"advantage_std={metrics['advantage_std']:.3f}"
    )

0it [00:00, ?it/s]

2it [00:00, 345.84it/s]

Autograd graph has been properly deallocated.


step 0: loss=-0.0000  mean_reward=27.188  advantage_std=0.926


0it [00:00, ?it/s]

...

2it [00:00, 740.78it/s]

Autograd graph has been properly deallocated.
step 3: loss=+0.0000  mean_reward=31.719  advantage_std=0.926


0it [00:00, ?it/s]

2it [00:00, 678.69it/s]

Autograd graph has been properly deallocated.
step 4: loss=-0.0000  mean_reward=34.188  advantage_std=0.926


## 5. Where the KeysAndValues innovation lives

The interesting part is step 5 of `grpo_step`: the policy gradient.

Instead of materializing per-token log-probs for the whole sequence and computing the GRPO loss externally (which would force autograd to retain activations for the entire completion), we express the GRPO loss as a `HeadModel` (`GRPOLossHeadModel`). The `LongContextGradientModel` then computes the gradient with **nested activation checkpointing** (outer loop over layers, inner over chunks), keeping GPU memory bounded regardless of completion length.

This is the same machinery that powers KeysAndValues' long-context supervised fine-tuning, now applied to RL.

In [6]:
from keys_values.finetune.grpo_loss import GRPOLossHeadModel

# The GRPO loss as a HeadModel: given logits, target tokens, old log-probs,
# and advantages, it computes the clipped surrogate loss chunk by chunk.
head = GRPOLossHeadModel(config, epsilon_low=0.2, epsilon_high=0.2)
print("GRPO loss head model:", head.NAME)
print("needs_logits:", head.needs_logits())
print("\nThis plugs into LongContextGradientModel exactly like")
print("CrossEntropyOnLogits does for supervised fine-tuning.")

GRPO loss head model: grpo_loss
needs_logits: True

This plugs into LongContextGradientModel exactly like
CrossEntropyOnLogits does for supervised fine-tuning.


## 6. Integration with HuggingFace TRL

Two ways to use these pieces with TRL:

**Log-probs override (merged, phase 1):** `keys_values.finetune.grpo.GRPOLongContextTrainer` subclasses TRL's `GRPOTrainer` and overrides `_get_per_token_logps_and_entropies` to use `compute_logprobs` for long sequences. Drop-in for TRL users who have a keys_values model.

**Standalone loop (this notebook, phase 2):** `grpo_step` runs the whole GRPO loop on a keys_values `GPT` directly, without TRL's `GRPOTrainer`. Useful when you want the KV-cache memory benefits across generation, scoring, *and* the gradient pass, without adapting a keys_values model to the HuggingFace `PreTrainedModel` interface.

The standalone loop reuses TRL-compatible building blocks (group-relative advantages, clipped surrogate loss), so the reward functions and data you'd write for TRL carry over.